In [ ]:
import os

for i in  range(10):
    dir_text = !dir
    if any([".venv" in text for text in dir_text]):
        print(".venv encontrado")
        break
    else:
        print(f'{i} - os.chdir("..")')
        os.chdir("..")


%load_ext autoreload
%autoreload 2

In [6]:
!dir

 O volume na unidade G � Armazenamento
 O N�mero de S�rie do Volume � 2091-BE76

 Pasta de g:\py_projects\fast-english

29/07/2025  18:47    <DIR>          .
27/07/2025  11:13    <DIR>          ..
26/07/2025  16:09                44 .gitattributes
03/07/2025  16:47    <DIR>          .github
26/07/2025  16:22             4.660 .gitignore
03/07/2025  16:47                 9 .python-version
03/07/2025  16:53    <DIR>          .venv
26/07/2025  16:22    <DIR>          add_new_data
29/07/2025  18:47    <DIR>          app
12/07/2025  20:33    <DIR>          database
03/07/2025  16:47    <DIR>          docs
03/07/2025  16:47           456.432 poetry.lock
03/07/2025  16:47             1.231 pyproject.toml
28/07/2025  11:21               208 README.md
28/07/2025  13:51               210 requirements.txt
26/07/2025  15:48            15.326 set_difference.txt
03/07/2025  16:47    <DIR>          tests
03/07/2025  16:47    <DIR>          trash
               8 arquivo(s)        478.120 bytes
      

TODO: Melhorias

Opção de selecionar e arrastar varias palavras
- 2 cliques em um lugar em branco e consegue selecionar 
- 2 cliques em uma palavra e é a cionado o audio dessa palavra

In [9]:
import tkinter as tk

class PalavraArrastavel:
    def __init__(self, canvas, texto, x, y):
        self.canvas = canvas
        self.texto = texto
        self.id = canvas.create_text(x, y, text=texto, font=("Arial", 20), anchor="nw")
        self.offset_x = 0
        self.offset_y = 0

        canvas.tag_bind(self.id, "<ButtonPress-1>", self.iniciar_arrasto)
        canvas.tag_bind(self.id, "<B1-Motion>", self.arrastar)
        canvas.tag_bind(self.id, "<ButtonRelease-1>", self.finalizar_arrasto)

    def iniciar_arrasto(self, event):
        self.offset_x = event.x - self.canvas.coords(self.id)[0]
        self.offset_y = event.y - self.canvas.coords(self.id)[1]

    def arrastar(self, event):
        novo_x = event.x - self.offset_x
        novo_y = event.y - self.offset_y
        self.canvas.coords(self.id, novo_x, novo_y)

    def finalizar_arrasto(self, event):
        pass  # Você pode adicionar lógica aqui se quiser

# Janela principal
root = tk.Tk()
root.title("Arraste as Palavras")

canvas = tk.Canvas(root, width=600, height=400, bg="white")
canvas.pack()

# Lista de palavras
palavras = ["the", "many", "is", "today"]
posicoes = [(50, 50), (150, 50), (250, 50), (350, 50)]

# Criar objetos arrastáveis
objetos = [PalavraArrastavel(canvas, palavra, x, y) for palavra, (x, y) in zip(palavras, posicoes)]

root.mainloop()

In [13]:
import tkinter as tk

class PalavraArrastavel:
    def __init__(self, canvas, texto, x, y):
        self.canvas = canvas
        self.texto = texto
        self.id = canvas.create_text(x, y, text=texto, font=("Arial", 20), anchor="nw")
        self.offset_x = 0
        self.offset_y = 0
        self.selecionada = False

        canvas.tag_bind(self.id, "<ButtonPress-1>", self.iniciar_arrasto)
        canvas.tag_bind(self.id, "<B1-Motion>", self.arrastar)
        canvas.tag_bind(self.id, "<ButtonRelease-1>", self.finalizar_arrasto)

    def iniciar_arrasto(self, event):
        if self.selecionada:
            self.offset_x = event.x
            self.offset_y = event.y

    def arrastar(self, event):
        if self.selecionada:
            dx = event.x - self.offset_x
            dy = event.y - self.offset_y
            for palavra in palavras:
                if palavra.selecionada:
                    x, y = canvas.coords(palavra.id)
                    canvas.coords(palavra.id, x + dx, y + dy)
            self.offset_x = event.x
            self.offset_y = event.y

    def finalizar_arrasto(self, event):
        pass

    def get_bbox(self):
        return canvas.bbox(self.id)

    def set_selecionada(self, estado):
        self.selecionada = estado
        cor = "blue" if estado else "black"
        canvas.itemconfig(self.id, fill=cor)

def iniciar_selecao(event):
    global selecao_ativa, retangulo_selecao
    selecao_ativa = True
    canvas.focus_set()
    retangulo_selecao = canvas.create_rectangle(event.x, event.y, event.x, event.y, outline="gray", dash=(2, 2))
    canvas.bind("<Motion>", atualizar_selecao)
    canvas.bind("<ButtonRelease-1>", finalizar_selecao)

def atualizar_selecao(event):
    global retangulo_selecao
    x1, y1, x2, y2 = canvas.coords(retangulo_selecao)
    canvas.coords(retangulo_selecao, x1, y1, event.x, event.y)

def finalizar_selecao(event):
    global selecao_ativa, retangulo_selecao
    selecao_ativa = False
    x1, y1, x2, y2 = canvas.coords(retangulo_selecao)
    x_min, x_max = min(x1, x2), max(x1, x2)
    y_min, y_max = min(y1, y2), max(y1, y2)

    for palavra in palavras:
        px1, py1, px2, py2 = palavra.get_bbox()
        if px1 >= x_min and px2 <= x_max and py1 >= y_min and py2 <= y_max:
            palavra.set_selecionada(True)
        else:
            palavra.set_selecionada(False)

    canvas.delete(retangulo_selecao)
    canvas.unbind("<Motion>")
    canvas.unbind("<ButtonRelease-1>")

# Janela principal
root = tk.Tk()
root.title("Seleção Múltipla de Palavras")

canvas = tk.Canvas(root, width=600, height=400, bg="white")
canvas.pack()

# Lista de palavras
textos = ["the", "many", "is", "today"]
posicoes = [(50, 50), (150, 50), (250, 50), (350, 50)]
palavras = [PalavraArrastavel(canvas, texto, x, y) for texto, (x, y) in zip(textos, posicoes)]

# Variáveis de seleção
selecao_ativa = False
retangulo_selecao = None

# Duplo clique para iniciar seleção
canvas.bind("<Double-Button-1>", iniciar_selecao)

root.mainloop()

---

In [10]:
!dir

guilherme


In [7]:
import tkinter as tk
import random

class WordBox(tk.Label):
    """Representa uma caixa de palavra arrastável."""
    def __init__(self, master_container, text, app_instance, **kwargs):
        super().__init__(master_container, text=text, borderwidth=2, relief="raised", 
                         padx=10, pady=5, bg="lightblue", font=("Arial", 10))
        self.app = app_instance
        self.original_bg = self.cget("bg")
        self.is_overlapping = False

        self.bind("<ButtonPress-1>", self.on_press)
        self.bind("<B1-Motion>", self.on_drag)
        self.bind("<ButtonRelease-1>", self.on_release)

        self._drag_offset_x = 0
        self._drag_offset_y = 0

    def on_press(self, event):
        """Chamado ao clicar na palavra."""
        self._drag_offset_x = event.x
        self._drag_offset_y = event.y
        self.lift() # Traz a palavra para frente na ordem de empilhamento

    def on_drag(self, event):
        """Chamado ao arrastar a palavra."""
        # Coordenadas do mouse relativas à tela
        # Coordenadas do container pai relativas à tela
        parent_root_x = self.master.winfo_rootx()
        parent_root_y = self.master.winfo_rooty()
        
        # Nova posição da palavra (canto superior esquerdo) relativa ao container pai
        new_x_in_parent = event.x_root - parent_root_x - self._drag_offset_x
        new_y_in_parent = event.y_root - parent_root_y - self._drag_offset_y
        
        self.place(x=new_x_in_parent, y=new_y_in_parent)
        self.app.check_all_collisions() # Verifica colisões continuamente

    def on_release(self, event):
        """Chamado ao soltar a palavra."""
        self.app.handle_drop(self) # App pode querer fazer algo ao soltar
        self.app.check_all_collisions() # Verificação final de colisão

    def update_overlap_visual(self, is_overlapping):
        """Atualiza a cor de fundo se houver sobreposição."""
        if is_overlapping != self.is_overlapping: # Evita reconfigurações desnecessárias
            self.is_overlapping = is_overlapping
            new_bg = "red" if self.is_overlapping else self.original_bg
            if self.cget("bg") != new_bg:
                 self.config(bg=new_bg)

class SentenceGameApp:
    """Classe principal da aplicação do jogo."""
    def __init__(self, master):
        self.master = master
        master.title("Forme a Frase Correta")
        master.geometry("800x600")

        # --- Configuração da Frase ---
        self.target_sentence_pt = "Isso é uma caneta"
        self.target_sentence_en = "this is a pen"
        
        # Alterne aqui para testar com a frase em português
        # self.current_target_sentence_list = self.target_sentence_pt.split()
        self.current_target_sentence_list = self.target_sentence_en.split()
        
        self.distractor_words_pt = ["gato", "casa", "bola", "azul"]
        self.distractor_words_en = ["dog", "cat", "house", "ball", "tree"]
        
        # Escolhe distratores baseados no idioma da frase (exemplo simples)
        current_distractors = self.distractor_words_en if self.current_target_sentence_list[0].isascii() else self.distractor_words_pt
        
        num_distractors = 3 # Quantidade de palavras distraidoras
        self.all_words_texts = self.current_target_sentence_list + random.sample(current_distractors, min(num_distractors, len(current_distractors)))
        random.shuffle(self.all_words_texts)

        # --- Área de Palavras Disponíveis (Visual) ---
        self.word_source_config = {"x": 10, "y": 30, "width": 780, "height": 150}
        tk.Label(master, text="Palavras disponíveis:", font=("Arial", 12)).place(x=self.word_source_config["x"], y=self.word_source_config["y"] - 25)
        self.word_source_frame = tk.Frame(master, borderwidth=2, relief="sunken")
        self.word_source_frame.place(x=self.word_source_config["x"], y=self.word_source_config["y"],
                                     width=self.word_source_config["width"], height=self.word_source_config["height"])

        # --- Zona de Montagem da Frase (Visual) ---
        drop_zone_label_y = self.word_source_config["y"] + self.word_source_config["height"] + 20
        tk.Label(master, text="Arraste as palavras aqui para formar a frase:", font=("Arial", 12)).place(x=10, y=drop_zone_label_y)
        
        self.drop_zone_config = {"x": 10, "y": drop_zone_label_y + 25, "width": 780, "height": 100}
        self.drop_zone_frame = tk.Frame(master, borderwidth=2, relief="sunken", bg="#e0e0e0") # Cor de fundo levemente cinza
        self.drop_zone_frame.place(x=self.drop_zone_config["x"], y=self.drop_zone_config["y"],
                                   width=self.drop_zone_config["width"], height=self.drop_zone_config["height"])
        
        self.word_boxes = [] # Lista para armazenar as instâncias de WordBox
        self.create_word_boxes()

        # --- Botão de Verificação ---
        verify_button_y = self.drop_zone_config["y"] + self.drop_zone_config["height"] + 30
        self.verify_button = tk.Button(master, text="Verificar Frase", command=self.verify_sentence, 
                                       font=("Arial", 12, "bold"), bg="lightgreen", relief="raised", borderwidth=3)
        self.verify_button.place(relx=0.5, y=verify_button_y, anchor=tk.CENTER)

        # --- Rótulo de Resultado ---
        result_label_y = verify_button_y + 50
        self.result_label = tk.Label(master, text="", font=("Arial", 14, "bold"))
        self.result_label.place(relx=0.5, y=result_label_y, anchor=tk.CENTER)

        self.master.update_idletasks() # Garante que as dimensões dos frames são calculadas
        self.check_all_collisions() # Verificação inicial (embora não devam colidir)

    def create_word_boxes(self):
        """Cria e posiciona as caixas de palavras na área de origem."""
        # As palavras são filhas de self.master (janela principal)
        # mas são posicionadas visualmente dentro da 'word_source_frame'
        
        # Coordenadas relativas a self.master para o posicionamento inicial
        start_x_abs = self.word_source_config["x"] + 10 # Padding interno
        start_y_abs = self.word_source_config["y"] + 10
        
        current_x = start_x_abs
        current_y = start_y_abs
        max_row_width = self.word_source_config["width"] - 20 # Considera padding bilateral
        row_height = 0

        for text in self.all_words_texts:
            box = WordBox(self.master, text=text, app_instance=self)
            box.update_idletasks() # Necessário para obter winfo_width/height corretos
            
            box_width = box.winfo_width()
            box_height = box.winfo_height()
            row_height = max(row_height, box_height) # Altura da linha atual

            if current_x + box_width > start_x_abs + max_row_width: # Se exceder a largura
                current_x = start_x_abs # Volta para o início da próxima linha
                current_y += row_height + 10 # Pula para a próxima linha
                row_height = box_height # Reseta a altura da linha

            box.place(x=current_x, y=current_y)
            self.word_boxes.append(box)
            current_x += box_width + 10 # Espaçamento entre palavras
        
        self.master.update_idletasks() # Garante que tudo foi posicionado

    def get_box_bounds(self, box_widget):
        """Retorna (x1, y1, x2, y2) da caixa relativo ao seu mestre (janela principal)."""
        box_widget.update_idletasks() # Garante dimensões atualizadas
        x1 = box_widget.winfo_x()
        y1 = box_widget.winfo_y()
        x2 = x1 + box_widget.winfo_width()
        y2 = y1 + box_widget.winfo_height()
        return x1, y1, x2, y2

    def check_all_collisions(self):
        """Verifica todas as caixas de palavras por sobreposições."""
        # Primeiro, reseta o estado visual de todas as caixas
        for box in self.word_boxes:
            box.update_overlap_visual(False)

        # Compara cada par de caixas
        for i in range(len(self.word_boxes)):
            for j in range(i + 1, len(self.word_boxes)):
                box1 = self.word_boxes[i]
                box2 = self.word_boxes[j]

                # Ignora caixas não visíveis (se aplicável no futuro)
                if not (box1.winfo_ismapped() and box2.winfo_ismapped()):
                    continue
                
                b1_x1, b1_y1, b1_x2, b1_y2 = self.get_box_bounds(box1)
                b2_x1, b2_y1, b2_x2, b2_y2 = self.get_box_bounds(box2)

                # Verifica sobreposição
                overlap_x = (b1_x1 < b2_x2) and (b1_x2 > b2_x1)
                overlap_y = (b1_y1 < b2_y2) and (b1_y2 > b2_y1)

                if overlap_x and overlap_y:
                    box1.update_overlap_visual(True)
                    box2.update_overlap_visual(True)

    def handle_drop(self, dropped_box):
        """Chamado quando uma caixa é solta. Pode ser usado para 'snap-to-grid' no futuro."""
        # No momento, a principal lógica de colisão já é tratada em on_drag e on_release.
        # Esta função é um placeholder para lógicas adicionais ao soltar, se necessário.
        pass # A verificação de colisão já será chamada em on_release.

    def verify_sentence(self):
        """Verifica se a frase montada na zona de montagem está correta."""
        words_in_drop_zone = []
        
        # Coordenadas da zona de montagem relativas à janela principal
        dz_x1 = self.drop_zone_config["x"]
        dz_y1 = self.drop_zone_config["y"]
        dz_x2 = dz_x1 + self.drop_zone_config["width"]
        dz_y2 = dz_y1 + self.drop_zone_config["height"]

        for box in self.word_boxes:
            if not box.winfo_ismapped(): continue

            b_x1, b_y1, b_x2, b_y2 = self.get_box_bounds(box)
            # Considera a caixa na zona se seu centro estiver dentro dela
            b_center_x = (b_x1 + b_x2) / 2
            b_center_y = (b_y1 + b_y2) / 2
            
            if (dz_x1 <= b_center_x <= dz_x2 and
                dz_y1 <= b_center_y <= dz_y2):
                # Adiciona (posição x, texto da palavra) para ordenação
                words_in_drop_zone.append((box.winfo_x(), box.cget("text")))
        
        # Ordena as palavras na zona de montagem pela sua posição X
        words_in_drop_zone.sort(key=lambda item: item[0])
        
        # Constrói a frase formada pelo usuário
        formed_sentence_list = [word_text for _, word_text in words_in_drop_zone]
        
        # Compara com a frase alvo
        if formed_sentence_list == self.current_target_sentence_list:
            self.result_label.config(text="Correto!", fg="green")
        else:
            self.result_label.config(text="Incorreto. Tente novamente.", fg="red")
        
        # Garante que o estado visual das colisões está atualizado
        self.check_all_collisions()

if __name__ == "__main__":
    root = tk.Tk()
    app = SentenceGameApp(root)
    root.mainloop()

In [8]:
import tkinter as tk
import random

class WordBox(tk.Label):
    """Caixa de palavra arrastável."""
    def __init__(self, master, text, app, **kwargs):
        super().__init__(
            master,
            text=text,
            borderwidth=2,
            relief="raised",
            padx=10,
            pady=5,
            bg="lightblue",
            font=("Arial", 10),
            **kwargs
        )
        self.app = app
        self.default_bg = self["bg"]
        self.drag_offset = (0, 0)
        self.is_overlapping = False

        self.bind("<ButtonPress-1>", self.start_drag)
        self.bind("<B1-Motion>", self.do_drag)
        self.bind("<ButtonRelease-1>", self.end_drag)

    def start_drag(self, event):
        self.lift()
        self.drag_offset = (event.x, event.y)

    def do_drag(self, event):
        new_x = event.x_root - self.master.winfo_rootx() - self.drag_offset[0]
        new_y = event.y_root - self.master.winfo_rooty() - self.drag_offset[1]
        self.place(x=new_x, y=new_y)
        self.app.check_all_collisions()

    def end_drag(self, event):
        self.app.handle_drop(self)
        self.app.check_all_collisions()

    def update_overlap_visual(self, overlapping):
        if overlapping != self.is_overlapping:
            self.is_overlapping = overlapping
            self.configure(bg="red" if overlapping else self.default_bg)


class SentenceGameApp:
    def __init__(self, master):
        self.master = master
        self.master.title("Forme a Frase Correta")
        self.master.geometry("800x600")

        self.target_sentences = {
            "pt": "Isso é uma caneta",
            "en": "this is a pen"
        }
        self.distractors = {
            "pt": ["gato", "casa", "bola", "azul"],
            "en": ["dog", "cat", "house", "ball", "tree"]
        }

        self.language = "pt"  # Troque para 'pt' se quiser
        self.target_words = self.target_sentences[self.language].split()
        self.word_pool = self.target_words + random.sample(self.distractors[self.language], 3)
        random.shuffle(self.word_pool)

        self.word_boxes = []

        self.create_widgets()
        self.master.after(100, self.check_all_collisions)

    def create_widgets(self):
        tk.Label(self.master, text="Palavras disponíveis:", font=("Arial", 12)).place(x=10, y=5)
        self.word_frame = tk.Frame(self.master, relief="sunken", borderwidth=2)
        self.word_frame.place(x=10, y=30, width=780, height=150)

        self.drop_label_y = 200
        tk.Label(self.master, text="Arraste as palavras aqui:", font=("Arial", 12)).place(x=10, y=self.drop_label_y)
        self.drop_frame = tk.Frame(self.master, bg="#eee", relief="sunken", borderwidth=2)
        self.drop_frame.place(x=10, y=self.drop_label_y + 25, width=780, height=100)

        self.create_word_boxes()

        self.verify_btn = tk.Button(self.master, text="Verificar Frase", bg="lightgreen", font=("Arial", 12, "bold"),
                                    command=self.verify_sentence)
        self.verify_btn.place(relx=0.5, y=360, anchor="center")

        self.result_label = tk.Label(self.master, font=("Arial", 14, "bold"))
        self.result_label.place(relx=0.5, y=410, anchor="center")

    def create_word_boxes(self):
        x, y = 20, 40
        max_width = 760
        row_height = 0

        for word in self.word_pool:
            box = WordBox(self.master, text=word, app=self)
            box.update_idletasks()
            w, h = box.winfo_width(), box.winfo_height()
            if x + w > max_width:
                x, y = 20, y + row_height + 10
                row_height = 0
            box.place(x=x, y=y)
            x += w + 10
            row_height = max(row_height, h)
            self.word_boxes.append(box)

    def handle_drop(self, dropped_box):
        pass  # Personalize esta função se desejar algo ao soltar

    def check_all_collisions(self):
        for box in self.word_boxes:
            box.update_overlap_visual(False)
        for i, box1 in enumerate(self.word_boxes):
            for box2 in self.word_boxes[i + 1:]:
                if self._is_overlapping(box1, box2):
                    box1.update_overlap_visual(True)
                    box2.update_overlap_visual(True)

    def _is_overlapping(self, box1, box2):
        x1, y1, x2, y2 = self.get_bounds(box1)
        a1, b1, a2, b2 = self.get_bounds(box2)
        return (x1 < a2 and x2 > a1) and (y1 < b2 and y2 > b1)

    def get_bounds(self, widget):
        widget.update_idletasks()
        x, y = widget.winfo_x(), widget.winfo_y()
        return x, y, x + widget.winfo_width(), y + widget.winfo_height()

    def verify_sentence(self):
        # Palavras ordenadas da drop zone
        drop_words = [
            box.cget("text")
            for box in sorted(
                self.word_boxes,
                key=lambda b: b.winfo_x()
            )
            if self.is_in_drop_zone(box)
        ]
        correct = drop_words == self.target_words
        self.result_label.config(
            text="✅ Correto!" if correct else "❌ Incorreto.",
            fg="green" if correct else "red"
        )

    def is_in_drop_zone(self, box):
        bx, by, bx2, by2 = self.get_bounds(box)
        dzx, dzy, dzx2, dzy2 = self.get_bounds(self.drop_frame)
        return (bx >= dzx and bx2 <= dzx2) and (by >= dzy and by2 <= dzy2)

if __name__ == "__main__":
    root = tk.Tk()
    app = SentenceGameApp(root)
    root.mainloop()
